# NutraMetrics: Разведочный анализ данных (EDA) и проверка статистических гипотез

**Автор:** R&D Formulation Chemist (6+ лет в FoodTech/Nutraceuticals *Nuversys* и Фармацевтике *Биннофарм Групп*) & Product Data Analyst

В данном ноутбуке проводится статистическое исследование **907 пероральных препаратов (БАДов и витаминов)**, собранных с живого аптечного рынка РФ (`Uteka.ru`, `Apteka.ru`, `Zdravcity.ru`, `Eapteka.ru`), и проверяются **5 продуктовых и химико-технологических гипотез** с использованием непараметрических критериев (`Kruskal-Wallis`, `Mann-Whitney U`, `Friedman`, `Wilcoxon`), поправки Холма–Бонферрони и двухфакторного дисперсионного анализа (`Type-II ANOVA`).

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

df = pd.read_csv('../data/processed/nutrametrics_priced.csv')
print(f'Размер очищенного датасета: {df.shape[0]} SKU x {df.shape[1]} признаков')
df[['nutra_category', 'price_rub', 'cost_per_day_rub', 'bap_dose_rub', 'nvi_score', 'brand_overpay_pct']].describe().round(2)

Размер очищенного датасета: 897 SKU x 79 признаков

       price_rub  cost_per_day_rub  bap_dose_rub  nvi_score  brand_overpay_pct
count     897.00            897.00        897.00     897.00             897.00
mean      832.50             24.68         20.79      58.65              16.39
std       815.07             31.50         29.60      19.56              72.17
min        50.00              0.21          0.16      14.00             -94.60
25%       349.00              7.77          6.28      45.70             -28.30
50%       580.00             15.43         11.86      59.20               0.00
75%      1023.00             29.25         24.60      72.70              44.90
max      8468.00            381.40        480.07      98.00             738.30

## H1. Парадокс биодоступности: R&D Tier 3 vs Tier 1/2 (C_day vs BAP-Dose & NVI)

- **Нулевая гипотеза ($H_0$):** H0: Номинальная цена дня курса (C_day), эффективная биодоступная цена (BAP-Dose) и индекс NVI одинаково распределены между Tier 1, Tier 2 и Tier 3.
- **Статистический критерий:** `Kruskal-Wallis H-test + попарный Mann-Whitney U (поправка Холма–Бонферрони)`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** Номинально день приема Tier 3 дороже Tier 1 (17.23 ₽ vs 10.3 ₽, H=50.8, p=9.27e-12). Однако за счет высокой биодоступности (K_bio=1.75) и адекватных дозировок эффективная стоимость BAP-Dose у Tier 3 (9.21 ₽, 95% ДИ [8.32; 11.58]) статистически значимо НИЖЕ, чем у Tier 2 (14.37 ₽, p_holm=7.77e-07) и выигрывает по среднему индексу ценности NVI (73.1 против 43.9 у Tier 1, H=206.7, p=1.34e-45).

In [2]:
# Результаты статистического теста H1
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H1']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H1",
  "title": "Парадокс биодоступности: R&D Tier 3 vs Tier 1/2 (C_day vs BAP-Dose & NVI)",
  "null_hypothesis": "H0: Номинальная цена дня курса (C_day), эффективная биодоступная цена (BAP-Dose) и индекс NVI одинаково распределены между Tier 1, Tier 2 и Tier 3.",
  "test_used": "Kruskal-Wallis H-test + попарный Mann-Whitney U (поправка Холма–Бонферрони)",
  "kruskal_c_day_H": 50.81,
  "kruskal_c_day_p": 9.26603407174069e-12,
  "kruskal_bap_H": 31.12,
  "kruskal_bap_p": 1.7468910724837706e-07,
  "kruskal_nvi_H": 206.65,
  "kruskal_nvi_p": 1.3363707844884084e-45,
  "epsilon_sq_nvi": 0.2306,
  "epsilon_sq_bap": 0.0347,
  "pairwise_bap_holm_p": {
    "Tier3_vs_Tier1": 0.25865260418939007,
    "Tier3_vs_Tier2": 7.773559396548165e-07,
    "Tier2_vs_Tier1": 0.0007879467733204645
  },
  "rank_biserial_t3_vs_t2": -0.2435,
  "group_medians_ci": {
    "Tier 1 (Basic)": {
      "n": 188,
      "c_day_median": 10.3,
      "c_day_ci95": [
        7.56,
        12.1
      ],
      "bap_me

## H2. Импортная премия vs локальное R&D-производство (контроль по Bio-Tier и категории)

- **Нулевая гипотеза ($H_0$):** H0: При фиксированном технологическом уровне (Bio-Tier) и категории вещества страна производства (Импорт vs РФ/ЕАЭС) не влияет на цену и индекс ценности NVI.
- **Статистический критерий:** `Двухфакторный OLS ANOVA (Type II SS) + Mann-Whitney U`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** Даже при строгом статистическом контроле категории вещества и уровня биодоступности (Tier 1–3) через Type-II ANOVA, импортные препараты несут чистую ценовую премию +133.4% (F=265.4, p=2.22e-52; медиана 1065.0 ₽ против 486.0 ₽ у РФ). При этом за счет высокой стоимости курса средний индекс потребительской ценности NVI у российских разработок значимо выше (63.6 против 44.9, p=5.85e-34).

In [3]:
# Результаты статистического теста H2
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H2']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H2",
  "title": "Импортная премия vs локальное R&D-производство (контроль по Bio-Tier и категории)",
  "null_hypothesis": "H0: При фиксированном технологическом уровне (Bio-Tier) и категории вещества страна производства (Импорт vs РФ/ЕАЭС) не влияет на цену и индекс ценности NVI.",
  "test_used": "Двухфакторный OLS ANOVA (Type II SS) + Mann-Whitney U",
  "mannwhitney_price_U": 119056.0,
  "mannwhitney_price_p": 7.451767165477421e-33,
  "mannwhitney_cday_p": 5.833826265039774e-24,
  "mannwhitney_nvi_p": 5.847105727007204e-34,
  "anova_import_F": 265.37,
  "anova_import_p": 2.2216717331191513e-52,
  "controlled_import_premium_pct": 133.4,
  "rank_biserial_price": 0.5223,
  "group_summary": {
    "Импортные БАДы": {
      "n": 237,
      "median_price_rub": 1065.0,
      "price_ci95": [
        940.0,
        1213.0
      ],
      "median_c_day_rub": 25.0,
      "mean_nvi": 44.9,
      "tier3_share_pct": 13.9
    },
    "Российские БАДы (РФ/ЕАЭС)": {
      "n": 660,
      "medi

## H3. Экономика галеновых форм доставки: Gummies и Липосомы против Таблеток и Капсул

- **Нулевая гипотеза ($H_0$):** H0: Галеновая форма выпуска (пастилки Gummies, липосомы/мицеллы, капсулы, таблетки) не влияет на стоимость суточного приема (C_day) и потребительский рейтинг.
- **Статистический критерий:** `Kruskal-Wallis H-test + Mann-Whitney U + Bootstrap 95% ДИ медианы`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** Галеновая матрица радикально меняет Unit-экономику (H=24.3, p=2.18e-05): жевательные пастилки Gummies обходятся потребителю в 11.16 ₽/день, а липосомальные/мицеллированные формы — в 33.2 ₽/день против 12.97 ₽/день у классических твердых таблеток (p=5.72e-01). При этом пастилки Gummies демонстрируют высокую приверженность потребителей, но в 62.1% случаев содержат сахар/патоку на первых местах состава — это открывает прямую R&D-нишу для бесcахарных микроинкапсулированных Gummies.

In [4]:
# Результаты статистического теста H3
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H3']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H3",
  "title": "Экономика галеновых форм доставки: Gummies и Липосомы против Таблеток и Капсул",
  "null_hypothesis": "H0: Галеновая форма выпуска (пастилки Gummies, липосомы/мицеллы, капсулы, таблетки) не влияет на стоимость суточного приема (C_day) и потребительский рейтинг.",
  "test_used": "Kruskal-Wallis H-test + Mann-Whitney U + Bootstrap 95% ДИ медианы",
  "kruskal_form_cday_H": 24.28,
  "kruskal_form_cday_p": 2.1804071070463022e-05,
  "mannwhitney_gummies_vs_tablets_cday_p": 0.5721820679282923,
  "mannwhitney_gummies_vs_tablets_rating_p": 0.024992212097567208,
  "form_metrics": {
    "Жевательные пастилки (Gummies)": {
      "n": 66,
      "c_day_median": 11.16,
      "c_day_ci95": [
        9.37,
        14.09
      ],
      "mean_bayesian_rating": 4.39,
      "sugar_share_pct": 62.1
    },
    "Липосомы / Мицеллы (R&D)": {
      "n": 22,
      "c_day_median": 33.2,
      "c_day_ci95": [
        25.0,
        54.08
      ],
      "mean_bayesian_rating": 4.24,
     

## H4. Технологический аудит эксципиентов: Диоксид титана (E171) в оболочках и Сахар в Gummies

- **Нулевая гипотеза ($H_0$):** H0: Частота присутствия диоксида титана (E171) и сахарно-паточных носителей независима от галеновой формы выпуска и страны происхождения.
- **Статистический критерий:** `Критерий Хи-квадрат Пирсона (χ²) + V Крамера + точный тест Фишера`
- **Вердикт:** **H0 отвергается (p < 0.0001 по обеим матрицам эксципиентов)**
- **R&D и Бизнес-вывод:** Выявлена строгая технологическая поляризация эксципиентов: 1) Диоксид титана E171 (запрещенный EFSA в ЕС с 2022 г.) сконцентрирован в пленочных покрытиях твердых таблеток (31.1% содержат E171) и непрозрачных твердых капсулах (14.3%), тогда как в порошках, пастилках и шипучих формах отсутствует (χ²=82.2, p=4.79e-15, V Крамера=0.303). 2) Сахар и глюкозно-паточные сиропы доминируют в жевательных пастилках Gummies (62.1% SKU) и шипучих таблетках (32.0%) против 2.3% в капсулах (χ²=253.0, p=6.30e-51, V Крамера=0.531).

In [5]:
# Результаты статистического теста H4
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H4']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H4",
  "title": "Технологический аудит эксципиентов: Диоксид титана (E171) в оболочках и Сахар в Gummies",
  "null_hypothesis": "H0: Частота присутствия диоксида титана (E171) и сахарно-паточных носителей независима от галеновой формы выпуска и страны происхождения.",
  "test_used": "Критерий Хи-квадрат Пирсона (χ²) + V Крамера + точный тест Фишера",
  "chi2_form_e171": 82.25,
  "p_form_e171": 4.78987543578631e-15,
  "cramers_v_form_e171": 0.3028,
  "chi2_form_sugar": 253.02,
  "p_form_sugar": 6.297001481619128e-51,
  "cramers_v_form_sugar": 0.5311,
  "p_fisher_origin_e171": 0.13674552199434636,
  "odds_ratio_origin_e171": 0.708,
  "form_excipient_profile": {
    "Жевательные пастилки (Gummies)": {
      "sku_count": 66,
      "e171_pct": 0.0,
      "sugar_pct": 62.1,
      "mean_k_clean": 0.953
    },
    "Жидкие формы / Капли / Спрей": {
      "sku_count": 76,
      "e171_pct": 5.3,
      "sugar_pct": 14.5,
      "mean_k_clean": 1.003
    },
    "Капсулы (Softgels / Vcaps)

## H5. Ценовой арбитраж между аптечными агрегаторами (Uteka, Apteka.ru, Zdravcity, Eapteka)

- **Нулевая гипотеза ($H_0$):** H0: Медианная цена идентичных препаратов (парные SKU) одинакова между агрегаторами Apteka.ru, Zdravcity.ru, Eapteka.ru и минимальной ценой Uteka.ru.
- **Статистический критерий:** `Парный непараметрический критерий Фридмана + парный тест Вилкоксона`
- **Вердикт:** **H0 отвергается (p < 0.0001)**
- **R&D и Бизнес-вывод:** На 129 полностью пересекающихся SKU критерий Фридмана фиксирует высокозначимое расхождение цен между агрегаторами (χ²_F=36.1, p=1.44e-08). Среди большой тройки федеральных дистрибьюторов медианная цена на Apteka.ru (951 ₽) статистически значимо ниже, чем на Zdravcity.ru (1105 ₽, p=3.91e-14) и Eapteka.ru (987 ₽, p=2.96e-05), а покупка через лучшую сеть мета-агрегатора Uteka (781 ₽) экономит в среднем 23.8% относительно среднерыночной аптечной полки.

In [6]:
# Результаты статистического теста H5
with open('../data/processed/hypothesis_results.json', encoding='utf-8') as f:
    h_res = json.load(f)['H5']
print(json.dumps(h_res, ensure_ascii=False, indent=2))

{
  "id": "H5",
  "title": "Ценовой арбитраж между аптечными агрегаторами (Uteka, Apteka.ru, Zdravcity, Eapteka)",
  "null_hypothesis": "H0: Медианная цена идентичных препаратов (парные SKU) одинакова между агрегаторами Apteka.ru, Zdravcity.ru, Eapteka.ru и минимальной ценой Uteka.ru.",
  "test_used": "Парный непараметрический критерий Фридмана + парный тест Вилкоксона",
  "matched_all3_skus": 129,
  "friedman_stat": 36.11,
  "friedman_p": 1.4419455007690877e-08,
  "wilcoxon_aptekaru_vs_zdravcity_p": 3.9107399833150615e-14,
  "wilcoxon_aptekaru_vs_eapteka_p": 2.963930445963307e-05,
  "wilcoxon_uteka_min_vs_market_avg_p": 6.307936140261464e-109,
  "matched_medians_rub": {
    "Uteka Min Price": 781.0,
    "Apteka.ru": 951.0,
    "Zdravcity.ru": 1105.0,
    "Eapteka.ru": 987.0,
    "Market Avg Price": 1208.0
  },
  "decision": "H0 отвергается (p < 0.0001)",
  "business_insight": "На 129 полностью пересекающихся SKU критерий Фридмана фиксирует высокозначимое расхождение цен между агрегато